# 하이퍼파라미터 Grid Search 실험

최적의 `chunk_size`, `overlap`, `k` 조합을 찾기 위한 실험입니다.

### 1: Chunk/Overlap 탐색 (`k=6` 고정)

- `(700, 70, 6)`
- `(900, 90, 6)`
- `(1000, 100, 6)`
- `(1200, 120, 6)`
- `(1500, 150, 6)`
- `(1000, 200, 6)`  (overlap 큰 케이스 확인)

### 2: `k` 튜닝 
1단계에서 성능이 좋은 상위 2개 조합에 대해 아래 `k`를 실험합니다.

- `k = [4, 6, 8, 10, 12]`

### 총 실험 수
- 1단계: 6회
- 2단계: 2 × 5 = 10회  ## 하이퍼파라미터 Grid Search 개요

현재 기준값은 `(chunk_size=1000, overlap=50, k=6)`입니다.  
이번 실험은 **효율적으로 최적 조합을 찾기 위한 2단계 방식**으로 진행합니다.

비교적 빠른 `k` 튜닝을 나중에 수행해 시간/비용 대비 효율을 높이는 전략입니다.


In [2]:
%load_ext autoreload
%autoreload 2

import os

# 1. macOS OpenMP 충돌 방지 (반드시 최상단 실행)
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# 2. 토크나이저 병렬 처리 충돌 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("환경 설정 완료. 이제 다음 셀들을 실행하세요.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
환경 설정 완료. 이제 다음 셀들을 실행하세요.


In [3]:
# [1] 공통 설정
import sys
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path('/Users/apple/Team2-RAG-Project')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(dotenv_path=PROJECT_ROOT / '.env')

from src.loader import load_documents_with_metadata
from src.embedding import split_documents, create_embeddings, build_faiss_vectorstore

CSV_PATH = PROJECT_ROOT / 'data/raw/data_list.csv'
PREPROCESSED_DIR = PROJECT_ROOT / 'data/preprocessed'
EVAL_PATH = PROJECT_ROOT / 'evaluation/test_questions.json'

dataset = json.loads(EVAL_PATH.read_text(encoding='utf-8'))
print('평가 문항 수:', len(dataset))
from src.retriever import build_hybrid_retriever as build_retriever_hybrid


/Users/apple/anaconda3/envs/Team2-RAG-Project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


평가 문항 수: 30


In [4]:
# [2] 문서 1회 로드
documents = load_documents_with_metadata(PREPROCESSED_DIR, CSV_PATH)
print('문서 수:', len(documents))


문서 수: 7547


In [5]:
# [3] 평가 함수(Recall@k, Hit@k, MRR)
KS = [1, 3, 5, 10]

def normalize_file_name(name):
    s = unicodedata.normalize('NFC', str(name or '')).lower().strip()
    s = re.sub(r'\.(pdf|hwp|hwpx|jsonl)$', '', s)
    s = re.sub(r'[\s_\-\(\)\[\]\{\}·ㆍ\.,!?:;`~|]', '', s)
    return s

def to_page_1based(page):
    try:
        p = int(page)
    except Exception:
        return None
    return p + 1 if p >= 0 else None

def gold_pairs(item):
    pairs = set()
    for ev in item.get('evidence', []):
        sf = normalize_file_name(ev.get('source_file'))
        pg = ev.get('page')
        if sf:
            pairs.add((sf, int(pg) if pg is not None else None))
    return pairs

def pred_pairs(docs):
    pairs = []
    for d in docs:
        md = getattr(d, 'metadata', {}) or {}
        sf = normalize_file_name(md.get('source_file') or md.get('source') or '')
        pg = to_page_1based(md.get('page'))
        pairs.append((sf, pg))
    return pairs

def evaluate_retriever(retriever, data, ks=KS):
    rows = []
    for item in data:
        q = item['question']
        gold = gold_pairs(item)
        docs = retriever.invoke(q)
        pred = pred_pairs(docs)

        row = {'id': item.get('id'), 'gold_evidence_count': len(gold)}

        rr = 0.0
        for rank, pair in enumerate(pred, start=1):
            if pair in gold:
                rr = 1.0 / rank
                break
        row['RR'] = rr

        for k in ks:
            topk = pred[:k]
            row[f'Hit@{k}'] = 1 if any(p in gold for p in topk) else 0
            if len(gold) == 0:
                row[f'Recall@{k}'] = 0.0
            else:
                row[f'Recall@{k}'] = len(set(topk) & gold) / len(gold)

        rows.append(row)

    detail_df = pd.DataFrame(rows)
    summary = {'MRR': detail_df['RR'].mean()}
    for k in ks:
        summary[f'Hit@{k}'] = detail_df[f'Hit@{k}'].mean()
        summary[f'Recall@{k}'] = detail_df[f'Recall@{k}'].mean()

    return summary, detail_df


In [6]:
# [4] 1단계: chunk/overlap 탐색 (k=6 고정)
STAGE1_GRID = [
    (700, 70, 6),
    (900, 90, 6),
    (1000, 100, 6),
    (1200, 120, 6),
    (1500, 150, 6),
    (1000, 200, 6),
]

embeddings = create_embeddings(model='text-embedding-3-small', chunk_size=100)
stage1_rows = []
cache = {}  # (chunk, overlap) -> {'split_docs', 'vectorstore'}

for chunk_size, overlap, k in STAGE1_GRID:
    print(f'[Stage1] chunk={chunk_size}, overlap={overlap}, k={k}')

    split_docs = split_documents(documents, chunk_size=chunk_size, chunk_overlap=overlap)
    vectorstore = build_faiss_vectorstore(split_docs, embeddings)

    retriever = build_retriever_hybrid(
        vectorstore=vectorstore,
        split_documents=split_docs,
        dense_k=k,
        sparse_k=k,
        dense_weight=0.7,
        sparse_weight=0.3,
    )

    summary, _ = evaluate_retriever(retriever, dataset, ks=KS)
    summary.update({'chunk_size': chunk_size, 'overlap': overlap, 'k': k, 'config': f'({chunk_size}, {overlap}, {k})'})
    stage1_rows.append(summary)

    cache[(chunk_size, overlap)] = {
        'split_docs': split_docs,
        'vectorstore': vectorstore,
    }

stage1_df = pd.DataFrame(stage1_rows)
stage1_df = stage1_df.sort_values('MRR', ascending=False).reset_index(drop=True)
stage1_df


[Stage1] chunk=700, overlap=70, k=6
[Stage1] chunk=900, overlap=90, k=6
[Stage1] chunk=1000, overlap=100, k=6
[Stage1] chunk=1200, overlap=120, k=6
[Stage1] chunk=1500, overlap=150, k=6
[Stage1] chunk=1000, overlap=200, k=6


,MRR,Hit@1,Recall@1,Hit@3,Recall@3,Hit@5,Recall@5,Hit@10,Recall@10,chunk_size,overlap,k,config
0,0.126984,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,0.233333,0.233333,700,70,6,"(700, 70, 6)"
1,0.124722,0.066667,0.066667,0.133333,0.133333,0.200000,0.200000,0.266667,0.266667,900,90,6,"(900, 90, 6)"
2,0.105873,0.066667,0.066667,0.100000,0.100000,0.133333,0.133333,0.233333,0.233333,1000,200,6,"(1000, 200, 6)"
3,0.104762,0.066667,0.066667,0.133333,0.133333,0.133333,0.133333,0.200000,0.200000,1000,100,6,"(1000, 100, 6)"
4,0.093056,0.066667,0.066667,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,1500,150,6,"(1500, 150, 6)"
5,0.071429,0.066667,0.066667,0.066667,0.066667,0.066667,0.066667,0.100000,0.100000,1200,120,6,"(1200, 120, 6)"


In [7]:
# [5] 2단계: 1단계 상위 2개 조합의 k 튜닝
K_GRID = [4, 6, 8, 10, 12]
top2 = stage1_df[['chunk_size', 'overlap']].head(2).to_dict('records')

stage2_rows = []
for combo in top2:
    chunk_size = int(combo['chunk_size'])
    overlap = int(combo['overlap'])
    assets = cache[(chunk_size, overlap)]

    for k in K_GRID:
        print(f'[Stage2] chunk={chunk_size}, overlap={overlap}, k={k}')

        retriever = build_retriever_hybrid(
            vectorstore=assets['vectorstore'],
            split_documents=assets['split_docs'],
            dense_k=k,
            sparse_k=k,
            dense_weight=0.7,
            sparse_weight=0.3,
        )

        summary, _ = evaluate_retriever(retriever, dataset, ks=KS)
        summary.update({'chunk_size': chunk_size, 'overlap': overlap, 'k': k, 'config': f'({chunk_size}, {overlap}, {k})'})
        stage2_rows.append(summary)

stage2_df = pd.DataFrame(stage2_rows)
stage2_df = stage2_df.sort_values('MRR', ascending=False).reset_index(drop=True)
stage2_df


[Stage2] chunk=700, overlap=70, k=4
[Stage2] chunk=700, overlap=70, k=6
[Stage2] chunk=700, overlap=70, k=8
[Stage2] chunk=700, overlap=70, k=10
[Stage2] chunk=700, overlap=70, k=12
[Stage2] chunk=900, overlap=90, k=4
[Stage2] chunk=900, overlap=90, k=6
[Stage2] chunk=900, overlap=90, k=8
[Stage2] chunk=900, overlap=90, k=10
[Stage2] chunk=900, overlap=90, k=12


,MRR,Hit@1,Recall@1,Hit@3,Recall@3,Hit@5,Recall@5,Hit@10,Recall@10,chunk_size,overlap,k,config
0,0.134247,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,0.266667,0.266667,700,70,12,"(700, 70, 12)"
1,0.131847,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,0.266667,0.266667,700,70,10,"(700, 70, 10)"
2,0.131086,0.066667,0.066667,0.133333,0.133333,0.200000,0.200000,0.300000,0.300000,900,90,10,"(900, 90, 10)"
3,0.130317,0.066667,0.066667,0.133333,0.133333,0.200000,0.200000,0.266667,0.266667,900,90,12,"(900, 90, 12)"
4,0.130093,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,0.266667,0.266667,700,70,8,"(700, 70, 8)"
5,0.128426,0.066667,0.066667,0.133333,0.133333,0.200000,0.200000,0.300000,0.300000,900,90,8,"(900, 90, 8)"
6,0.126984,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,0.233333,0.233333,700,70,6,"(700, 70, 6)"
7,0.124722,0.066667,0.066667,0.133333,0.133333,0.200000,0.200000,0.266667,0.266667,900,90,6,"(900, 90, 6)"
8,0.122222,0.100000,0.100000,0.100000,0.100000,0.166667,0.166667,0.200000,0.200000,700,70,4,"(700, 70, 4)"
9,0.113889,0.066667,0.066667,0.133333,0.133333,0.166667,0.166667,0.200000,0.200000,900,90,4,"(900, 90, 4)"


In [8]:
# [6] 최종 best 조합 확인 + 저장
best = stage2_df.iloc[0].to_dict()
print('Best config:', {k: best[k] for k in ['chunk_size', 'overlap', 'k']})
print('Best MRR:', round(best['MRR'], 6))

out_dir = PROJECT_ROOT / 'evaluation'
out_dir.mkdir(parents=True, exist_ok=True)
stage1_df.to_csv(out_dir / 'gridsearch_stage1.csv', index=False)
stage2_df.to_csv(out_dir / 'gridsearch_stage2.csv', index=False)
print('saved:', out_dir / 'gridsearch_stage1.csv')
print('saved:', out_dir / 'gridsearch_stage2.csv')


Best config: {'chunk_size': 700, 'overlap': 70, 'k': 12}
Best MRR: 0.134247
saved: /Users/apple/Team2-RAG-Project/evaluation/gridsearch_stage1.csv
saved: /Users/apple/Team2-RAG-Project/evaluation/gridsearch_stage2.csv


### 결론

- 1단계 실험에서 `chunk_size=700, overlap=70`과 `chunk_size=900, overlap=90`이 상위 성능을 보였고,  
  큰 chunk(`1200+`)는 전반적으로 성능이 하락했습니다.
- 2단계 `k` 튜닝 결과,  
  - `(700, 70)`에서는 `k=12`가 최고  
  - `(900, 90)`에서는 `k=10`이 최고였습니다.
- **최종 MRR 기준 최적 조합은 `(700, 70, 12)`** 입니다.
- 단, **Hit@10/Recall@10 최대는 `(900, 90, 10)`** 이므로,  
  운영 목표(초기 순위 정확도 vs 상위 K 회수율)에 따라 보조 후보로 유지합니다.


### Advanced 자산 로드 실험 (faiss_advanced + split_documents_advanced.pkl)

In [9]:
# [7] Advanced 자산 로드 후 평가
import pickle
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from src.embedding import create_embeddings

ADV_DB_PATH = PROJECT_ROOT / "data/vectorstore/faiss_advanced"
ADV_PKL_PATH = PROJECT_ROOT / "data/vectorstore/split_documents_advanced.pkl"

embeddings = create_embeddings(model="text-embedding-3-small", chunk_size=100)
vectorstore_adv = FAISS.load_local(
    str(ADV_DB_PATH),
    embeddings,
    allow_dangerous_deserialization=True,
)

with open(ADV_PKL_PATH, "rb") as f:
    split_docs_adv = pickle.load(f)

dense_retriever = vectorstore_adv.as_retriever(search_kwargs={"k": 6})
sparse_retriever = BM25Retriever.from_documents(split_docs_adv)
sparse_retriever.k = 6

retriever_adv = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)

summary_adv, detail_adv = evaluate_retriever(retriever_adv, dataset, ks=KS)
print("Advanced 자산 평가 요약:")
print(summary_adv)
print("문항별 결과 shape:", detail_adv.shape)


Advanced 자산 평가 요약:
{'MRR': np.float64(0.13), 'Hit@1': np.float64(0.06666666666666667), 'Recall@1': np.float64(0.06666666666666667), 'Hit@3': np.float64(0.16666666666666666), 'Recall@3': np.float64(0.16666666666666666), 'Hit@5': np.float64(0.2), 'Recall@5': np.float64(0.2), 'Hit@10': np.float64(0.26666666666666666), 'Recall@10': np.float64(0.26666666666666666)}
문항별 결과 shape: (30, 11)
